# RAG Pipeline Components

**Module:** 04 — RAG

Walk each stage of a production RAG pipeline with contracts, demos, and failure modes.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Name every stage from query intake to validated response
- Explain what each stage consumes and emits
- Implement toy versions of rewrite, retrieve, rerank, pack, and cite
- Choose metrics per stage for production monitoring


## Input Query

**Definition.** The raw user question plus channel metadata (locale, tenant, product).

**Why it matters.** Everything downstream is conditioned on this string and its implicit intent.

**How it works.** Validate length/language; attach session/tenant; optionally classify intent.

**Intuition.** Same words mean different things in billing vs engineering portals.

**Common pitfalls.**
- Trusting raw user text as safe instructions
- Dropping channel metadata early

**When to use.** Every request—normalize before retrieval.

```mermaid
flowchart LR
  A[Input Query] --> B[Query Processing]
  B --> C[Embedding]
  C --> D[Vector Search]
  D --> E[Reranking]
  E --> F[Context Selection]
  F --> G[Prompt Construction]
  G --> H[LLM Response]
```


In [ ]:
# Demo 1 — normalize inbound query envelope
from dataclasses import dataclass

@dataclass
class QueryEnvelope:
    text: str
    tenant_id: str
    locale: str
    channel: str
    product: str | None = None

    def normalized(self):
        return QueryEnvelope(self.text.strip(), self.tenant_id, self.locale.lower(),
                             self.channel, self.product)

q = QueryEnvelope("  refund window? ", "acme", "EN-us", "web", "shop").normalized()
print(q)


In [ ]:
# Demo 2 — guardrails on query size / emptiness
def validate_query(text, max_chars=2000):
    t = text.strip()
    if not t: raise ValueError("empty query")
    if len(t) > max_chars: raise ValueError("too long")
    return t
for s in ["", "ok", "x"*2005]:
    try: print(repr(s[:20]), "->", validate_query(s)[:20])
    except ValueError as e: print(repr(s[:20]), "->", e)


### Try it yourself — Input Query

1. Write a one-sentence SLA for the 'Input Query' stage.
2. Name one metric you would log for 'Input Query'.
3. Describe a failure that looks like an LLM bug but is really 'Input Query'.


## Query Processing

**Definition.** Transforms that improve retrieval: rewrite, expand, hyphenate SKUs, translate, HyDE.

**Why it matters.** Users type telegraphic, ambiguous, or multi-intent questions.

**How it works.** Rule cleanups + optional LLM rewrite; keep original for lexical search.

**Intuition.** Help the librarian interpret a mumbled request before searching shelves.

**Common pitfalls.**
- Rewrites that drift from user intent
- Expanding so broadly recall becomes noise

**When to use.** When paraphrase mismatch shows up in eval failures.


In [ ]:
# Demo 1 — lightweight rewrite rules
import re
def process_query(q: str) -> dict:
    original = q.strip()
    q2 = re.sub(r"\brefunds?\b", "refund policy", original, flags=re.I)
    q2 = re.sub(r"\s+", " ", q2)
    return {"original": original, "processed": q2, "sku": re.findall(r"SKU[- ]?\d+", original, re.I)}
print(process_query("refund for SKU-42 please"))


In [ ]:
# Demo 2 — multi-query expansion (template stand-in for LLM)
def expand(q):
    return [q, f"policy about {q}", f"how to {q}"]
print(expand("reset password"))


In [ ]:
# Demo 3 — HyDE-style hypothetical document (offline sketch)
def hyde_doc(question: str) -> str:
    return f"This document explains '{question}' with definitions, steps, and exceptions."
print(hyde_doc("international shipping duties"))


### Try it yourself — Query Processing

1. Write a one-sentence SLA for the 'Query Processing' stage.
2. Name one metric you would log for 'Query Processing'.
3. Describe a failure that looks like an LLM bug but is really 'Query Processing'.


## Embedding

**Definition.** Map query (and chunks) into a shared vector space with a chosen embedder.

**Why it matters.** Dense retrieval quality is bounded by embedding model + domain fit.

**How it works.** Same model/version/dim at index and query; normalize if using cosine.

**Intuition.** Meaning becomes geometry; distance approximates relatedness.

**Common pitfalls.**
- Mixing models
- Embedding uncleaned queries with UI chrome

**When to use.** Any dense or hybrid retriever.


In [ ]:
# Demo 1 — bag embedding + cosine (stand-in for API embedder)
import numpy as np

def embed(text, vocab):
    toks = text.lower().split()
    v = np.array([toks.count(w) for w in vocab], float)
    return v / (np.linalg.norm(v) + 1e-9)

vocab = ["refund", "shipping", "password", "days", "policy"]
q = embed("refund policy days", vocab)
print(q.round(3))


In [ ]:
# Demo 2 — embedding API request shape
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {"model": "text-embedding-3-small", "input": ["refund window?", "Refunds within 60 days."]}
print(json.dumps(req, indent=2))
print("header Authorization: Bearer", YOUR_API_KEY)
# expected response shape:
print({"data": [{"embedding": [0.01, -0.02], "index": 0}], "usage": {"total_tokens": 12}})


### Try it yourself — Embedding

1. Write a one-sentence SLA for the 'Embedding' stage.
2. Name one metric you would log for 'Embedding'.
3. Describe a failure that looks like an LLM bug but is really 'Embedding'.


## Vector Search

**Definition.** Approximate nearest neighbor lookup returning top-k chunk candidates.

**Why it matters.** This is the first cut of evidence; errors here cannot be fixed by prose.

**How it works.** ANN index (HNSW/IVF...) with optional metadata prefilters.

**Intuition.** Find neighbors in meaning-space, not spelling-space.

**Common pitfalls.**
- k too small
- No filters in multi-tenant indexes
- Stale index

**When to use.** Default first-stage retrieval.


In [ ]:
# Demo 1 — brute-force top-k
import numpy as np
docs = ["refund sixty days", "ship three days", "reset password settings"]
vocab = sorted({w for d in docs for w in d.split()})
def emb(t):
    v = np.array([t.split().count(w) for w in vocab], float); return v/(np.linalg.norm(v)+1e-9)
M = np.stack([emb(d) for d in docs])
scores = M @ emb("refund days")
for i in np.argsort(-scores)[:2]:
    print(i, float(scores[i]), docs[i])


In [ ]:
# Demo 2 — metadata prefilter then search
rows = [
    {"id": "1", "tenant": "acme", "vec": [1.0, 0.0]},
    {"id": "2", "tenant": "beta", "vec": [0.9, 0.1]},
]
def search(rows, tenant, qvec, k=1):
    import numpy as np
    cand = [r for r in rows if r["tenant"]==tenant]
    scored = sorted(((float(np.dot(r["vec"], qvec)), r) for r in cand), reverse=True)
    return scored[:k]
print(search(rows, "acme", [1.0, 0.0]))


### Try it yourself — Vector Search

1. Write a one-sentence SLA for the 'Vector Search' stage.
2. Name one metric you would log for 'Vector Search'.
3. Describe a failure that looks like an LLM bug but is really 'Vector Search'.


## Reranking

**Definition.** A cross-encoder or LLM re-scores a candidate list for precision.

**Why it matters.** Bi-encoders are fast but coarse; rerankers recover precision cheaply on top-n.

**How it works.** Retrieve k=50, rerank to top 5–10 for packing.

**Intuition.** Skim many, carefully read few.

**Common pitfalls.**
- Reranking tiny k
- Ignoring latency budgets

**When to use.** When recall@50 is fine but precision@5 is not.


In [ ]:
# Demo 1 — toy cross-score via token overlap
def rerank(query, candidates):
    q = set(query.lower().split())
    scored = [(len(q & set(c.lower().split())), c) for c in candidates]
    return sorted(scored, reverse=True)
print(rerank("refund window", ["shipping times", "refund within 60 days", "password help"]))


In [ ]:
# Demo 2 — retrieve wide, keep narrow
candidates = [f"doc-{i}" for i in range(50)]
reranked = candidates[:50]  # pretend scores
packed = reranked[:5]
print("retrieved", len(candidates), "packed", packed)


### Try it yourself — Reranking

1. Write a one-sentence SLA for the 'Reranking' stage.
2. Name one metric you would log for 'Reranking'.
3. Describe a failure that looks like an LLM bug but is really 'Reranking'.


## Context Selection

**Definition.** Packing policy: which chunks enter the prompt under a token budget.

**Why it matters.** Window space is scarce; duplicates and low scores waste it.

**How it works.** Dedupe, diversity, score thresholds, parent expansion, budget packer.

**Intuition.** Curate a briefing binder, not a junk drawer.

**Common pitfalls.**
- Naive top-k concat
- No score floor

**When to use.** Always—define an explicit packer.


In [ ]:
# Demo 1 — budget packer
def pack(chunks, budget_chars=120):
    out, used = [], 0
    for c in chunks:
        if used + len(c) > budget_chars: break
        out.append(c); used += len(c) + 1
    return out, used
chunks = ["Refunds 60 days.", "Keep receipt.", "Shipping 3-5 days.", "More policy text here."]
print(pack(chunks))


In [ ]:
# Demo 2 — dedupe near duplicates
def dedupe(chunks, thresh=0.5):
    out = []
    for c in chunks:
        ws = set(c.lower().split())
        if any(len(ws & set(o.lower().split()))/max(1,len(ws)) >= thresh for o in out):
            continue
        out.append(c)
    return out
print(dedupe(["refunds within 60 days", "refunds within sixty days", "shipping"]))


### Try it yourself — Context Selection

1. Write a one-sentence SLA for the 'Context Selection' stage.
2. Name one metric you would log for 'Context Selection'.
3. Describe a failure that looks like an LLM bug but is really 'Context Selection'.


## Prompt Construction

**Definition.** Assemble system rules, evidence blocks, citation format, and user question.

**Why it matters.** Prompt contract determines faithfulness and refusal behavior.

**How it works.** Delimit evidence; instruct cite-or-refuse; separate untrusted user text.

**Intuition.** Job description + briefing + acceptance tests.

**Common pitfalls.**
- Evidence after a huge chat history
- No citation scheme

**When to use.** Every grounded generation call.


In [ ]:
# Demo 1 — evidence-delimited prompt
def build_prompt(question, contexts):
    blocks = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    return (
        "Use ONLY the EVIDENCE. Cite [n]. If missing, say you do not know.\n\n"
        f"EVIDENCE:\n{blocks}\n\nQUESTION:\n{question}"
    )
print(build_prompt("Refund window?", ["Within 60 days."]))


In [ ]:
# Demo 2 — messages array for chat APIs
import json
msgs = [
    {"role": "system", "content": "Ground answers in evidence. Refuse if insufficient."},
    {"role": "user", "content": "EVIDENCE:\n[1] 60 days\n\nQUESTION:\nRefund window?"},
]
print(json.dumps(msgs, indent=2))


### Try it yourself — Prompt Construction

1. Write a one-sentence SLA for the 'Prompt Construction' stage.
2. Name one metric you would log for 'Prompt Construction'.
3. Describe a failure that looks like an LLM bug but is really 'Prompt Construction'.


## LLM Response

**Definition.** Model output plus post-checks: citations present, policy filters, structured parse.

**Why it matters.** Users see this; it must be useful and safe even when retrieval is imperfect.

**How it works.** Generate at low temperature for FAQ; validate schema; attach sources UI-side.

**Intuition.** The intern writes the memo using only the binder you gave them.

**Common pitfalls.**
- High temperature on policy QA
- No post-validation

**When to use.** Production answers that others will trust.


In [ ]:
# Demo 1 — parse citations from model text
import re
def extract_cites(text):
    return re.findall(r"\[(\d+)\]", text)
ans = "You may refund within 60 days [1]. Shipping is separate [2]."
print(extract_cites(ans))


In [ ]:
# Demo 2 — post-validate faithfulness proxy
def validate(answer, contexts):
    if "do not know" in answer.lower(): return {"ok": True, "reason": "refusal"}
    cites = extract_cites(answer)
    if not cites: return {"ok": False, "reason": "missing citations"}
    if any(int(c) < 1 or int(c) > len(contexts) for c in cites):
        return {"ok": False, "reason": "bad citation"}
    return {"ok": True, "reason": "ok"}
print(validate(ans, ["60 days", "3-5 days"]))


In [ ]:
# Demo 3 — API response shape (placeholder)
import json
YOUR_API_KEY = "YOUR_API_KEY"
response = {
  "id": "chatcmpl_demo",
  "choices": [{"message": {"role": "assistant", "content": "Refunds within 60 days [1]."},
               "finish_reason": "stop"}],
  "usage": {"prompt_tokens": 220, "completion_tokens": 18},
}
print(json.dumps(response, indent=2))
print("called with key", YOUR_API_KEY[:8]+"...")


### Try it yourself — LLM Response

1. Write a one-sentence SLA for the 'LLM Response' stage.
2. Name one metric you would log for 'LLM Response'.
3. Describe a failure that looks like an LLM bug but is really 'LLM Response'.


## Glossary

- **ANN**: Approximate nearest neighbor search
- **packer**: Budgeted context selector


## Summary & Key Takeaways

- Pipelines fail at stages—instrument each one
- Retrieve wide, rerank/pack narrow
- Prompt contracts enforce cite-or-refuse
- Post-validation catches silent grounding failures

### Practice

Trace one real question through all eight stages on paper, then in code stubs.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
